# 06 — Reproducibility Audit

**Purpose:** Verify that the entire pipeline is reproducible end-to-end.  
Checks:
1. All leakage guards pass on the final outputs
2. Two simulation runs with the same seed produce bit-identical win probabilities
3. All 48 canonical team names resolve correctly through all three translation maps
4. The feature table is stable across re-runs (no random imputation)
5. The training data time-order is strictly chronological
6. The freeze manifest is consistent with the loaded datasets

This notebook is intentionally **read-only** with respect to `outputs/` — it asserts,  
not produces.

---
Run this notebook after all earlier notebooks have completed to validate the pipeline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import warnings
import joblib

from src.leakage_guard import (
    FREEZE_DATE, ELO_SNAPSHOT_DATE, FORM_WINDOW_END,
    check_freeze_date, check_elo_snapshot, check_no_synthetic_data,
    check_training_rows_chronological, check_canonical_names,
    check_annex_c, run_all_checks, LeakageError,
)
from src.name_map import CANONICAL_48, TO_CANONICAL, WC_DEBUTANTS, canonicalize, assert_all_canonical
from src.features import load_ds2, load_ds1
from src.simulation import run_tournament_simulation, build_md3_schedule
from src.features import load_ds16, load_ds17

DATA_ROOT = PROJECT_ROOT
ARC_BASE  = DATA_ROOT / "archive.zip"
ARC2      = DATA_ROOT / "archive (2).zip"
ARC4      = DATA_ROOT / "archive (4).zip"
OUTPUTS   = PROJECT_ROOT / "outputs"
ANNEX_C   = DATA_ROOT / "third_place_annex_c.csv"

RESULTS = {}  # accumulate pass/fail per check

def _check(name, fn):
    try:
        fn()
        RESULTS[name] = "PASS"
        print(f"  ✓ {name}")
    except (AssertionError, LeakageError, Exception) as e:
        RESULTS[name] = f"FAIL: {e}"
        print(f"  ✗ {name}: {e}")

print(f"Freeze date  : {FREEZE_DATE}")
print(f"Elo snapshot : {ELO_SNAPSHOT_DATE}")
print(f"Form window  : ...–{FORM_WINDOW_END}")
print()

## Check 1 — Leakage guards on raw data

In [ ]:
print("=== Leakage Guards — Raw Data ===")

ds2 = load_ds2(ARC4)
ds1 = load_ds1(ARC2)

_check("DS1 freeze date",    lambda: check_freeze_date(ds1, date_column="date"))
_check("DS2 Elo snapshot",   lambda: check_elo_snapshot(ds2))

## Check 2 — Leakage guards on pipeline outputs

In [ ]:
print("=== Leakage Guards — Pipeline Outputs ===")

team_features_path  = OUTPUTS / "team_features_freeze.parquet"
training_rows_path  = OUTPUTS / "training_rows.parquet"

if team_features_path.exists():
    tf = pd.read_parquet(team_features_path)
    _check("No synthetic data in team_features", lambda: check_no_synthetic_data(tf))
else:
    print("  SKIP: team_features_freeze.parquet not found — run notebook 02 first")

if training_rows_path.exists():
    tr = pd.read_parquet(training_rows_path)
    _check("Training rows chronological", lambda: check_training_rows_chronological(tr))
    _check("No synthetic data in training_rows", lambda: check_no_synthetic_data(tr))
else:
    print("  SKIP: training_rows.parquet not found — run notebook 02 first")

if ANNEX_C.exists():
    annex_c_df = pd.read_csv(ANNEX_C)
    _check("Annex C schema", lambda: check_annex_c(annex_c_df))
else:
    print("  SKIP: third_place_annex_c.csv not found")

## Check 3 — Name mapping completeness

In [ ]:
print("=== Name Mapping Completeness ===")

def _name_map_checks():
    assert len(CANONICAL_48) == 48, f"Expected 48 teams, got {len(CANONICAL_48)}"
    failures = [t for t in CANONICAL_48 if canonicalize(t) != t]
    assert not failures, f"Round-trip failures: {failures}"
    assert_all_canonical(list(CANONICAL_48), context="CANONICAL_48")

def _debutant_checks():
    for team in WC_DEBUTANTS:
        assert team in CANONICAL_48, f"{team} in WC_DEBUTANTS but not in CANONICAL_48"

def _variant_map_checks():
    for variant, canonical in TO_CANONICAL.items():
        assert canonical in CANONICAL_48, (
            f"TO_CANONICAL[{variant!r}] = {canonical!r} not in CANONICAL_48"
        )

_check("48 canonical teams",        _name_map_checks)
_check("Debutant set canonical",     _debutant_checks)
_check("TO_CANONICAL values valid",  _variant_map_checks)

print(f"  TO_CANONICAL has {len(TO_CANONICAL)} variant entries")

## Check 4 — Feature table stability

In [ ]:
print("=== Feature Table Stability ===")

if team_features_path.exists():
    tf1 = pd.read_parquet(team_features_path)
    tf2 = pd.read_parquet(team_features_path)

    def _stability_check():
        numeric_cols = tf1.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if tf1[col].notna().any():
                diff = (tf1[col] - tf2[col]).abs().max()
                assert diff == 0.0, f"Column {col} differs between two loads: max diff = {diff}"

    _check("Feature table stable across two reads", _stability_check)

    def _no_nulls_in_key_features():
        key_features = [c for c in ["elo_rating", "elo_win_expectancy"] if c in tf1.columns]
        for feat in key_features:
            n_null = tf1[feat].isnull().sum()
            assert n_null == 0, f"{feat} has {n_null} nulls in team_features"

    _check("Key features have no nulls", _no_nulls_in_key_features)
else:
    print("  SKIP: team_features_freeze.parquet not found")

## Check 5 — Simulation reproducibility

In [ ]:
print("=== Simulation Reproducibility ===")

models_ready = all([
    (OUTPUTS / "group_stage_model.joblib").exists(),
    (OUTPUTS / "knockout_model.joblib").exists(),
    (OUTPUTS / "penalty_model.joblib").exists(),
    team_features_path.exists(),
])

if models_ready:
    group_model   = joblib.load(OUTPUTS / "group_stage_model.joblib")
    knockout_model= joblib.load(OUTPUTS / "knockout_model.joblib")
    penalty_model = joblib.load(OUTPUTS / "penalty_model.joblib")
    tf = pd.read_parquet(team_features_path)

    ds16 = load_ds16(ARC_BASE)
    ds17 = load_ds17(ARC_BASE)
    md3_schedule = build_md3_schedule(ds16, ds17)

    common_kwargs = dict(
        n_simulations=200,
        group_model=group_model,
        knockout_model=knockout_model,
        penalty_model=penalty_model,
        team_features=tf,
        md3_schedule=md3_schedule,
        ds16=ds16,
        ds17=ds17,
        annex_c_path=ANNEX_C,
        seed=42,
    )

    print("  Running two 200-simulation runs with seed=42...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r1 = run_tournament_simulation(**common_kwargs)
        r2 = run_tournament_simulation(**common_kwargs)

    def _reproducibility_check():
        wp1 = r1.win_probabilities
        wp2 = r2.win_probabilities
        if isinstance(wp1, dict):
            s1, s2 = pd.Series(wp1), pd.Series(wp2)
        else:
            s1, s2 = wp1, wp2
        diff = (s1 - s2).abs().max()
        assert diff == 0.0, f"Two runs with seed=42 differ by {diff:.2e}"

    _check("Two runs with seed=42 are identical", _reproducibility_check)

    # Verify probability invariants on run 1
    def _prob_invariants():
        wp = r1.win_probabilities
        s = pd.Series(wp) if isinstance(wp, dict) else wp
        total = s.sum()
        assert abs(total - 1.0) < 1e-4, f"Win probs sum to {total}, not 1.0"
        assert (s >= 0).all(), f"Negative win probs found"
        assert len(s) == 48, f"Expected 48 teams in output, got {len(s)}"

    _check("Win probs sum to 1.0 and are non-negative", _prob_invariants)

else:
    print("  SKIP: models or feature table not found — run notebooks 02 and 03 first")

## Check 6 — Training data chronological order

In [ ]:
print("=== Training Data Time Order ===")

if training_rows_path.exists():
    tr = pd.read_parquet(training_rows_path)

    def _no_future_elo():
        check_training_rows_chronological(tr)

    def _year_ceiling():
        if "wc_year" in tr.columns:
            max_year = int(tr["wc_year"].max())
            assert max_year <= 2022, (
                f"Training rows contain wc_year={max_year} > 2022 (freeze ceiling)"
            )

    def _outcome_distribution():
        if "outcome" in tr.columns:
            counts = tr["outcome"].value_counts()
            total = len(tr)
            assert total == 896, f"Expected 896 training rows, got {total}"
            # WIN + LOSS should be approximately equal (mirrored perspective)
            win_n  = int(counts.get(2, 0))
            loss_n = int(counts.get(0, 0))
            assert abs(win_n - loss_n) <= 10, (
                f"WIN ({win_n}) and LOSS ({loss_n}) counts should mirror each other"
            )

    _check("No elo_year_used >= match_year", _no_future_elo)
    _check("Max wc_year <= 2022",            _year_ceiling)
    _check("896 training rows, WIN≈LOSS",    _outcome_distribution)
else:
    print("  SKIP: training_rows.parquet not found")

## Check 7 — Model serialisation round-trip

In [ ]:
import io, pickle

print("=== Model Serialisation Round-Trip ===")

for model_file in ["group_stage_model.joblib", "knockout_model.joblib", "penalty_model.joblib"]:
    path = OUTPUTS / model_file
    if not path.exists():
        print(f"  SKIP: {model_file} not found")
        continue

    def _roundtrip(p=path):
        model = joblib.load(p)
        buf = io.BytesIO()
        joblib.dump(model, buf)
        buf.seek(0)
        model2 = joblib.load(buf)
        # Both objects should have the same type
        assert type(model) == type(model2), (
            f"Type mismatch after roundtrip: {type(model)} vs {type(model2)}"
        )

    _check(f"{model_file} round-trip", _roundtrip)

## Check 8 — Output file inventory

In [ ]:
print("=== Output File Inventory ===")

expected_outputs = [
    # Notebook 01
    "group_standings_freeze.csv",
    "third_place_ranking_freeze.csv",
    # Notebook 02
    "team_features_freeze.parquet",
    "training_rows.parquet",
    "feature_audit_report.csv",
    # Notebook 03
    "group_stage_model.joblib",
    "knockout_model.joblib",
    "penalty_model.joblib",
    # Notebook 04
    "bayesian_update_table.csv",
    "win_probabilities.csv",
    "simulation_log.parquet",
    # Notebook 05
    "final_summary_table.csv",
]

expected_charts = [
    "charts/calibration_curve.png",
    "charts/update_shifts.png",
    "charts/win_probability_chart.png",
    "charts/group_standings_heatmap.png",
    "charts/feature_importance_chart.png",
    "charts/bayesian_shifts_chart.png",
]

inventory_rows = []
for fname in expected_outputs + expected_charts:
    path = OUTPUTS / fname
    exists = path.exists()
    size_kb = path.stat().st_size // 1024 if exists else None
    inventory_rows.append({"file": fname, "exists": exists, "size_kb": size_kb})

inv_df = pd.DataFrame(inventory_rows)
n_present = inv_df["exists"].sum()
n_total   = len(inv_df)
print(f"  {n_present}/{n_total} expected output files present")
inv_df

## Final Report

In [ ]:
passed = sum(1 for v in RESULTS.values() if v == "PASS")
failed = [(k, v) for k, v in RESULTS.items() if v != "PASS"]

print("\n" + "=" * 60)
print(f"  Reproducibility Audit: {passed}/{len(RESULTS)} checks passed")
print("=" * 60)

if failed:
    print("\nFailed checks:")
    for name, msg in failed:
        print(f"  ✗ {name}")
        print(f"      {msg}")
else:
    print("\n  All checks passed — pipeline is reproducible and leakage-free.")

# Final assertion: no leakage failures permitted
leakage_failures = [v for v in RESULTS.values() if "LeakageError" in str(v) or "FAIL" in v and "leakage" in v.lower()]
assert not leakage_failures, (
    f"Leakage detected in {len(leakage_failures)} check(s). Review failures above."
)